In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/small_bench/checkpoints/pre_cell_20.pickle

In [ ]:
%%cudf.pandas.profile
### cell 20 ###

def add_gap_rows_2(essay):
    import numpy as np
    import pandas as pd

    cols_to_keep = ['discourse_start', 'discourse_end', 'discourse_type', 'gap_length', 'gap_end_length']
    # pull only the relevant rows and columns
    df_essay = train.query(f"id == '{essay}'")[cols_to_keep].reset_index(drop=True)
    print(df_essay)

    # vectorized computation of gaps between segments
    # shift the end positions down by one to align with the next row
    prev_ends = df_essay['discourse_end'].shift(1)
    # only consider rows 1..N-1 where gap_length>0
    mask_between = (df_essay['gap_length'] > 0) & (df_essay.index > 0)
    if mask_between.any():
        df_between = df_essay.loc[mask_between, []].copy()
        df_between['discourse_start'] = prev_ends[mask_between] + 1
        df_between['discourse_end']   = df_essay.loc[mask_between, 'discourse_start'] - 1
        df_between['discourse_type']  = 'Nothing'
        df_between['gap_length']      = np.nan
        df_between['gap_end_length']  = np.nan
        # append the new gap rows
        df_essay = pd.concat([df_essay, df_between], ignore_index=True)

    # sort all segments by start
    df_essay = df_essay.sort_values('discourse_start').reset_index(drop=True)

    # handle a trailing gap at the end if present
    last_gap_end = df_essay['gap_end_length'].iloc[-1]
    if last_gap_end > 0:
        start = df_essay['discourse_end'].iloc[-1] + 1
        end = start + last_gap_end
        df_tail = pd.DataFrame({
            'discourse_start': [start],
            'discourse_end':   [end],
            'discourse_type':  ['Nothing'],
            'gap_length':      [np.nan],
            'gap_end_length':  [np.nan]
        })
        df_essay = pd.concat([df_essay, df_tail], ignore_index=True)

    return df_essay
